# Week 1-2

In [ ]:
import math

# ── CHANGE ONLY THIS ──────────────────────
RIM_RADIUS   = 0.27  # metres  ← your one parameter
# ─────────────────────────────────────────

N            = 16
CENTER       = (6.1, 0, 2.98)
TUBE_RADIUS  = 0.02
RGBA         = "1.0 1.0 1.0 0.7"
CONAFFINITY  = "1"
CONTYPE      = "0"

seg_half = math.pi * RIM_RADIUS / N
cx, cy, cz = CENTER

print(f"<!-- Rim: R={RIM_RADIUS}, N={N}, tube={TUBE_RADIUS} -->")
for i in range(N):
    theta = 2 * math.pi * i / N
    px = cx + RIM_RADIUS * math.cos(theta)
    py = cy + RIM_RADIUS * math.sin(theta)
    ax = -math.cos(theta)
    ay = -math.sin(theta)
    print(
        f'<geom type="cylinder" '
        f'size="{TUBE_RADIUS} {seg_half:.5f}" '
        f'pos="{px:.4f} {py:.4f} {cz}" '
        f'axisangle="{ax:.4f} {ay:.4f} 0 1.5708" '
        f'conaffinity="{CONAFFINITY}" contype="{CONTYPE}" '
        f'rgba="{RGBA}"/>'
    )

<!-- Rim: R=0.27, N=16, tube=0.02 -->
<geom type="cylinder" size="0.02 0.05301" pos="6.3700 0.0000 2.95" axisangle="-1.0000 -0.0000 0 1.5708" conaffinity="1" contype="0" rgba="1.0 1.0 1.0 0.7"/>
<geom type="cylinder" size="0.02 0.05301" pos="6.3494 0.1033 2.95" axisangle="-0.9239 -0.3827 0 1.5708" conaffinity="1" contype="0" rgba="1.0 1.0 1.0 0.7"/>
<geom type="cylinder" size="0.02 0.05301" pos="6.2909 0.1909 2.95" axisangle="-0.7071 -0.7071 0 1.5708" conaffinity="1" contype="0" rgba="1.0 1.0 1.0 0.7"/>
<geom type="cylinder" size="0.02 0.05301" pos="6.2033 0.2494 2.95" axisangle="-0.3827 -0.9239 0 1.5708" conaffinity="1" contype="0" rgba="1.0 1.0 1.0 0.7"/>
<geom type="cylinder" size="0.02 0.05301" pos="6.1000 0.2700 2.95" axisangle="-0.0000 -1.0000 0 1.5708" conaffinity="1" contype="0" rgba="1.0 1.0 1.0 0.7"/>
<geom type="cylinder" size="0.02 0.05301" pos="5.9967 0.2494 2.95" axisangle="0.3827 -0.9239 0 1.5708" conaffinity="1" contype="0" rgba="1.0 1.0 1.0 0.7"/>
<geom type="cylinder"

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Reacher-v5", xml_file =r"C:\vsr\Naren\FAFO-RL\agnet_xml\reacher.xml", render_mode = "human")
observation, info = env.reset(seed=42)


print("Obs shape:", observation.shape)
print("Action space:", env.action_space)

try:
    for i in range(1000):  # just 5 steps to see info
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i} | reward: {reward:.4f} | info: {info}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()

In [ ]:
import mujoco
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Pendulum-v1")
model = SAC("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=20000)
print("Stack working")

In [ ]:
env = gym.make("Pendulum-v1", render_mode = "human")


obs, info = env.reset(seed=42)

try:
    total_reward = 0
    for i in range(1000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
            print(total_reward)
            observation, info = env.reset()
            total_reward = 0
finally:
    env.close()

## Reacher SAC

In [ ]:
import numpy as np
import gymnasium as gym
class CustomPendulumEnv(gym.Wrapper):
    def angle_normalize(self, x):
        return ((x + np.pi) % (2 * np.pi)) - np.pi

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # obs = [cos(th), sin(th), thdot]
        th = np.arctan2(obs[1], obs[0])   # recover angle from cos/sin
        thdot = obs[2]
        u = np.clip(action, -2.0, 2.0)[0]

        # your custom reward
        costs = self.angle_normalize(th)**2 + 0.1 * thdot**2
        modified_reward = -costs           # dropped action cost term

        return obs, modified_reward, terminated, truncated, info

In [ ]:
import inspect
# import gymnasium as gym

# Create the base environment (unwrapped to bypass time limits/wrappers)
env = CustomPendulumEnv(gym.make("Pendulum-v1"))

# Print the source code of the step function where the reward is calculated
print(inspect.getsource(env.step))

In [ ]:
from stable_baselines3 import SAC, PPO
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=20e3, tb_log_name="pendulum_modifield_reward")
print("Stack working")

## Different Seed Testing

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO
env = gym.make("Reacher-v5")
# get_device('cpu')
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=500e3, tb_log_name="reacher_sac_long")
# model = SAC("MlpPolicy", env, verbose=1,seed = 33, device="cpu", tensorboard_log = r".\log")
# model.learn(total_timesteps=20e3, tb_log_name="pendulum_different_seed")
print("Stack working")

In [ ]:
model.save(r".\models\reacher_sac_500k")

In [ ]:
del model

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import SAC

# class MountainCarWrapper(gym.Wrapper):
#     def step(self, action):
#         obs, reward, terminated, truncated, info = self.env.step(action)
#         position, velocity = obs
#         reward += 10 * abs(velocity)
#         reward += 3 * (position + 0.5)
#         return obs, reward, terminated, truncated, info

# env = MountainCarWrapper(gym.make("MountainCarContinuous-v0"))
env = gym.make("MountainCarContinuous-v0")
model = SAC("MlpPolicy", env,
            learning_starts=1000,
            verbose=1,
            tensorboard_log=r".\log",
            seed=42)
model.learn(total_timesteps=100_000,
            tb_log_name="SAC_MountainCar_normal")

In [ ]:
# import mujoco
import gymnasium as gym
from stable_baselines3 import SAC, PPO

model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\log")
model.learn(total_timesteps=100e3, tb_log_name="mountain_car_continuous_reward_wrapper")
print("Stack working")

In [ ]:
# import gymnasium as gym
# from stable_baselines3 import SAC, PPO

env = gym.make("MountainCarContinuous-v0", render_mode = "human")
# model = SAC.load(r".\models\reacher_sac_500k", env=env)

# obs, info = env.reset(seed=32)
obs, info = env.reset()

try:
    total_reward = 0
    for i in range(10000):  # just 5 steps to see info
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        # print(f"Step {i} | reward: {reward:.4f} | info: {info}")
        total_reward += reward
        if terminated or truncated:
        # print(total_reward)
            observation, info = env.reset()
        total_reward = 0
finally:
    env.close()

# Week 3

In [ ]:
import mujoco

model = mujoco.MjModel.from_xml_path(
    r"E:\personal projects\FAFO-RL\agnet_xml\three_pointer.xml"
)
data = mujoco.MjData(model)
print("Bodies  :", model.nbody)
print("Joints  :", model.njnt)
print("Actuators:", model.nu)
print("qpos shape:", data.qpos.shape)
print("Valid ✓")

In [ ]:
# import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC
from pathlib import Path
from stable_baselines3.common.env_checker import check_env
# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)
# current_dir = Path.cwd()
# xml_path_str = str(current_dir.parent / "agnet_xml" / "three_pointer.xml")
env = gym.make(
    "ThreePointer-v0",
    # xml_file=xml_path_str,
    render_mode="human",
    
)
check_env(env)
# observation, info = env.reset(seed=42)
# print("Obs shape:", observation.shape)
# print("Action space:", env.action_space)

# try:
#     for i in range(10000):
#         action = env.action_space.sample()
#         observation, reward, terminated, truncated, info = env.step(action)
#         print(f"{i}", f"Observation {observation[7]}")

#         if terminated or truncated:
#             observation, info = env.reset()
# finally:
#     env.close()

In [ ]:
# import mujoco
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.env_util import make_vec_env
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC
from pathlib import Path

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).

register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    render_mode="human"
)
model = SAC("MlpPolicy", env, verbose=1, device="cpu", tensorboard_log = r".\three_pointer_log",seed=3, learning_starts= 500)
model.learn(total_timesteps=500e3, tb_log_name="default_settings")
print("Stack working")

In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
import gymnasium as gym
from gymnasium.envs.registration import register

register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

# training env — no render
env =  gym.make("ThreePointer-v0", render_mode = 'human')

# eval env — separate
# eval_env = gym.make("ThreePointer-v0")

# eval_callback = EvalCallback(
#     env,
#     best_model_save_path="./best_model/",
#     log_path="./logs/",
#     eval_freq=10_000,
#     n_eval_episodes=10,
#     deterministic=True,
#     verbose=1
# )

model = SAC(
    "MlpPolicy",
    env,
    verbose=1,
    device="cuda",
    tensorboard_log="./three_pointer_log",
    seed=3,
    learning_starts=5_000,
    # learning_rate=1e-4,       # conservative for contact physics
    # gamma=0.995,              # high — throw reward is delayed
    # batch_size=256,
    # buffer_size=500_000,
    # ent_coef="auto",
    # policy_kwargs=dict(net_arch=[256, 256]),
)

model.learn(
    total_timesteps=500_000,
    tb_log_name="fixed_reward_v1",
    # callback=eval_callback
)

model.save("three_pointer_500k")
print("Done")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./three_pointer_log\fixed_reward_v1_8
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 27.2     |
|    ep_rew_mean     | -233     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 54       |
|    time_elapsed    | 2        |
|    total_timesteps | 109      |
| train/             |          |
|    actor_loss      | 0.782    |
|    critic_loss     | 222      |
|    ent_coef        | 0.998    |
|    ent_coef_loss   | -0.007   |
|    learning_rate   | 0.0003   |
|    n_updates       | 8        |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 24.9     |
|    ep_rew_mean     | -214     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 55       |
|    time_elapsed    | 3        |
|    to

KeyboardInterrupt: 

In [ ]:
model.save(r".\models\three_pointer_sac_500k_terminatedlessthansuccess")


In [ ]:
import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    render_mode="human",
)
model = SAC.load(r"E:\personal projects\FAFO-RL\scripts\best_model\best_model.zip", env=env)
observation, info = env.reset()          # unpack the tuple — this was the crash
try:
    for i in range(10000):
        action, _states = model.predict(observation, deterministic=True)
        observation, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            print(f'{i}' ,f"terminated{terminated}", f"truncated {truncated}")
            observation, info = env.reset()
finally:
    env.close()

In [ ]:
import os
import gymnasium as gym
from gymnasium.envs.registration import register
from stable_baselines3 import SAC

# Do this once, before gym.make — entry_point is "module_name:ClassName".
# reacher_env.py must be importable (same folder as this script, or on sys.path).
register(
    id="ThreePointer-v0",
    entry_point="three_pointer_env:ThreePointerEnv",
    max_episode_steps=500,
)

env = gym.make(
    "ThreePointer-v0",
    render_mode="human",
)
observation, info = env.reset()   

try:
    for i in range(1000):  # just 5 steps to see info
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)
        print(f"Step {i} | terminated: {terminated:.4f} | observation: {observation[4]} | Reward {reward}")

        if terminated or truncated:
            observation, info = env.reset()
finally:
    env.close()